# Day 2-01｜Detection Box Footpoint 與 BEV 投影互動
> Python 籃球運動資料分析課程  
> 本單元先以已訓練 YOLO detector 產生球員框，再調整 bbox 觀察 bottom-center footpoint 對 BEV 投影的影響。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 從模型輸出的 player bbox 取得 bottom-center footpoint。
- 使用 Day 1 Homography 將 footpoint 投影到 BEV。
- 透過互動工具調整 bbox，觀察 BEV 點位變化。


## 課程流程
1. 對球場 frame 執行 YOLO detection。
2. 顯示含有 player box 與 footpoint 的影像。
3. 使用互動 UI 調整 bbox，觀察 footpoint 與 BEV 投影。


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("找不到 src/course_setup.py，請確認目前目錄位於課程 repo 內。")

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT)


In [ ]:
import json
from src.cv_utils import (
    bottom_center,
    draw_boxes,
    draw_points,
    read_image_rgb,
    save_image_rgb,
    show_image,
    side_by_side,
)
from src.geometry_utils import compute_homography, project_points, render_bev_court
from src.yolo_utils import (
    COURT_KEYPOINT_DISPLAY_NAMES,
    bev_court_bounds,
    court_vertices_bev_in_bounds,
    detector_model_path,
    run_detector_on_image,
)

IMAGE_PATH = COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png"
BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"
frame = read_image_rgb(IMAGE_PATH)
bev = render_bev_court(BEV_SPEC_PATH)

bev_points_all = court_vertices_bev_in_bounds(bev_court_bounds(BEV_SPEC_PATH))
bev_lookup = {
    name: [float(x), float(y)]
    for name, (x, y) in zip(COURT_KEYPOINT_DISPLAY_NAMES, bev_points_all)
}

HOMOGRAPHY_PAIR_JSON = r"""
{
  "pairs": [
    {"keypoint_name": "right_paint_ft_top", "camera_xy": [818.0, 480.0]},
    {"keypoint_name": "right_free_throw", "camera_xy": [1216.0, 453.0]},
    {"keypoint_name": "right_paint_ft_bottom", "camera_xy": [1410.0, 657.0]},
    {"keypoint_name": "right_paint_baseline_bottom", "camera_xy": [672.0, 623.0]},
    {"keypoint_name": "right_baseline_top", "camera_xy": [674.0, 423.0]},
    {"keypoint_name": "right_baseline_bottom", "camera_xy": [1434.0, 424.0]}
  ]
}
"""

raw_pair_data = json.loads(HOMOGRAPHY_PAIR_JSON)
pairs = []
for raw_pair in raw_pair_data["pairs"]:
    keypoint_name = str(raw_pair["keypoint_name"])
    if "bev_xy" in raw_pair:
        bev_xy = [float(v) for v in raw_pair["bev_xy"]]
    else:
        if keypoint_name not in bev_lookup:
            raise KeyError(f"找不到 BEV keypoint: {keypoint_name}")
        bev_xy = [float(v) for v in bev_lookup[keypoint_name]]
    pairs.append(
        {
            "keypoint_name": keypoint_name,
            "camera_xy": [float(v) for v in raw_pair["camera_xy"]],
            "bev_xy": bev_xy,
        }
    )

H = compute_homography(
    [pair["camera_xy"] for pair in pairs],
    [pair["bev_xy"] for pair in pairs],
)

detections, _ = run_detector_on_image(
    detector_model_path(COURSE_ROOT),
    frame,
    conf=0.25,
    imgsz=960,
)
players = [d for d in detections if d.class_name == "player"]
if not players:
    raise RuntimeError("此 frame 沒有偵測到 player，請調整 conf 或更換影像。")

player = players[0]
bbox = player.bbox_xyxy
foot = bottom_center(bbox)
bev_foot = project_points([foot], H)[0]

frame_vis = draw_boxes(frame, [bbox], [f"player {player.confidence:.2f}"])
frame_vis = draw_points(frame_vis, [foot], ["footpoint"])
bev_vis = draw_points(bev, [bev_foot], ["projected footpoint"])
preview = side_by_side(frame_vis, bev_vis)
show_image(preview, "Detection box and projected BEV footpoint")

out_path = COURSE_ROOT / "assets" / "results" / "d2_01_manual_box_to_bev.png"
save_image_rgb(out_path, preview)
print("saved:", out_path)


In [ ]:
from src.notebook_widgets import show_bbox_to_bev_tool

show_bbox_to_bev_tool(
    IMAGE_PATH,
    initial_bbox=bbox,
    homography_matrix=H.tolist(),
    bev_width=bev.shape[1],
    bev_height=bev.shape[0],
    canvas_width=1000,
    title="調整 player bbox 並觀察 BEV footpoint",
)


課堂操作：調整 `x1`、`y1`、`x2`、`y2`。bbox 底邊中心點越接近球員真正的地面接觸位置，BEV 投影越穩定。